In [1]:
import os, time, numpy as np, pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# Paths
train_path = "../Sentiment/sentiment_model_data/train.csv"
save_root = "../Sentiment/sentiment_model"
os.makedirs(save_root, exist_ok=True)

print("✅ Training file:", train_path)



✅ Training file: ../Sentiment/sentiment_model_data/train.csv


In [2]:
df = pd.read_csv(train_path).dropna()
print("✅ Loaded dataset:", df.shape)
print(df["label"].value_counts())

# Split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(f"Train size: {len(train_texts)} | Validation size: {len(val_texts)}")


✅ Loaded dataset: (102992, 2)
label
0    40693
2    34805
1    27494
Name: count, dtype: int64
Train size: 82393 | Validation size: 20599


In [3]:
def tokenize_data(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def tokenize(batch):
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)
    train_ds = Dataset.from_dict({"text": train_texts, "label": train_labels})
    val_ds   = Dataset.from_dict({"text": val_texts, "label": val_labels})
    train_ds = train_ds.map(tokenize, batched=True)
    val_ds   = val_ds.map(tokenize, batched=True)
    train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return tokenizer, train_ds, val_ds

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro"),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro")
    }


In [4]:
def train_and_evaluate(model_name, epochs=2):
    print(f"\n🚀 Training {model_name}")
    tokenizer, train_ds, val_ds = tokenize_data(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

    args = TrainingArguments(
        output_dir=os.path.join(save_root, model_name.replace("/", "_")),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        save_total_limit=1,
        logging_steps=100,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )

    start = time.time()
    trainer.train()
    train_time = time.time() - start

    results = trainer.evaluate()
    print(f"\n📊 {model_name} Results:")
    for k,v in results.items():
        print(f"{k}: {v:.4f}")
    print(f"⏱️ Training time: {train_time/60:.2f} min")

    # Save
    model_dir = os.path.join(save_root, model_name.replace("/", "_"), "final")
    os.makedirs(model_dir, exist_ok=True)
    model.save_pretrained(model_dir)
    tokenizer.save_pretrained(model_dir)

    return {"model": model_name, **results, "train_time": train_time, "path": model_dir}


In [5]:
# -------------------------------
# Cell: helpers for resuming + persistence
# -------------------------------
import os, json, time
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline

# reuse save_root from earlier; if not present, set explicitly
try:
    save_root
except NameError:
    save_root = "../Sentiment/sentiment_model"
os.makedirs(save_root, exist_ok=True)

results_file = os.path.join(save_root, "results_all.json")

def _results_load():
    if os.path.exists(results_file):
        with open(results_file, "r") as f:
            return json.load(f)
    # check if there's a results_all variable in current session
    if "results_all" in globals():
        return globals()["results_all"]
    return []

def _results_save(results):
    with open(results_file, "w") as f:
        json.dump(results, f, indent=2)

def saved_model_final_dir(model_name):
    """Replicate the earlier save naming: model_name.replace('/', '_') + '/final'"""
    return os.path.join(save_root, model_name.replace("/", "_"), "final")

# load existing results_all (if any)
results_all = _results_load()
print("Loaded results entries:", len(results_all))


Loaded results entries: 0


In [6]:
# -------------------------------
# Cell: Evaluate a model already saved on disk (no retrain)
# -------------------------------
from datasets import Dataset

def evaluate_saved_model(model_name):
    """
    Loads a saved model+tokenizer from disk and runs evaluation on val_ds.
    Returns dict with model name, eval metrics, train_time=0 (since no training here), path.
    """
    model_dir = saved_model_final_dir(model_name)
    if not os.path.isdir(model_dir):
        print(f"✖ Saved model not found at {model_dir}")
        return None

    print(f"🔎 Loading saved model from: {model_dir}")
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)

    # Reuse tokenize_data to create train_ds/val_ds (tokenize_data uses the model_name string to choose tokenizer
    # but here we already have tokenizer; safer to re-tokenize using our tokenizer)
    def tokenize_with_existing(ds_texts):
        def _tok(batch):
            return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)
        ds = Dataset.from_dict({"text": ds_texts, "label": val_labels})  # we only need val labels here
        ds = ds.map(_tok, batched=True)
        ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
        return ds

    # NOTE: assumes train_texts/val_texts/train_labels/val_labels are already defined earlier in notebook
    try:
        val_ds  # just check exists
    except NameError:
        # if those variables aren't in scope (e.g., you restarted kernel), rebuild splits
        print("val_ds not in memory — rebuilding splits from CSV")
        df = pd.read_csv(train_path).dropna()
        train_texts, val_texts, train_labels, val_labels = train_test_split(
            df["text"].tolist(),
            df["label"].tolist(),
            test_size=0.2,
            random_state=42,
            stratify=df["label"]
        )
    # build val dataset with tokenizer loaded from disk
    # val_labels must exist in scope now
    val_ds_for_eval = Dataset.from_dict({"text": val_texts, "label": val_labels}).map(
        lambda batch: tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128),
        batched=True
    )
    val_ds_for_eval.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

    # Build a Trainer only for evaluation (no train_dataset)
    args = TrainingArguments(
        output_dir=os.path.join(save_root, "tmp_eval"),
        per_device_eval_batch_size=32,
        do_train=False,
        logging_strategy="no",
        report_to="none"
    )
    trainer = Trainer(
        model=model,
        args=args,
        eval_dataset=val_ds_for_eval,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics
    )
    results = trainer.evaluate()
    # Normalize to consistent keys used later in comparison
    entry = {
        "model": model_name,
        "eval_loss": float(results.get("eval_loss", None)) if results.get("eval_loss", None) is not None else None,
        "eval_accuracy": float(results.get("eval_accuracy", None)) if results.get("eval_accuracy", None) is not None else None,
        "eval_f1": float(results.get("eval_f1", None)) if results.get("eval_f1", None) is not None else None,
        "eval_precision": float(results.get("eval_precision", None)) if results.get("eval_precision", None) is not None else None,
        "eval_recall": float(results.get("eval_recall", None)) if results.get("eval_recall", None) is not None else None,
        "train_time": 0.0,
        "path": model_dir
    }
    print(f"✅ Evaluation complete for {model_name}")
    return entry

# Example: evaluate distilbert if saved
distil_entry = evaluate_saved_model("distilbert-base-uncased")
if distil_entry:
    # avoid duplicate entries if already present (by model name)
    if not any(r["model"] == distil_entry["model"] for r in results_all):
        results_all.append(distil_entry)
        _results_save(results_all)
        print("Appended DistilBERT evaluation to results_all and saved.")
    else:
        print("DistilBERT entry already present in results_all; skipping append.")


🔎 Loading saved model from: ../Sentiment/sentiment_model\distilbert-base-uncased\final
val_ds not in memory — rebuilding splits from CSV


Map:   0%|          | 0/20599 [00:00<?, ? examples/s]

C:\Users\keshu\AppData\Local\Temp\ipykernel_23756\2139211328.py:60: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


✅ Evaluation complete for distilbert-base-uncased
Appended DistilBERT evaluation to results_all and saved.


In [7]:
# results_all = []
# results_all.append(train_and_evaluate("distilbert-base-uncased", epochs=2))     # Accuracy-first


In [8]:
# import json, glob, os

# outdir = "../Sentiment/sentiment_model/distilbert-base-uncased"  # training output dir
# trainer_state = os.path.join(outdir, "trainer_state.json")
# best_metrics = {}
# if os.path.exists(trainer_state):
#     with open(trainer_state, "r") as f:
#         ts = json.load(f)
#     # trainer_state contains `log_history` with evaluation entries; find best by f1
#     logs = ts.get("log_history", [])
#     # extract eval entries
#     eval_entries = [e for e in logs if "eval_loss" in e or "eval_f1" in e]
#     # pick last eval entry or highest f1
#     if eval_entries:
#         last = eval_entries[-1]
#         best_metrics = {
#             "model": "distilbert-base-uncased",
#             "eval_accuracy": last.get("eval_accuracy"),
#             "eval_f1": last.get("eval_f1"),
#             "eval_precision": last.get("eval_precision"),
#             "eval_recall": last.get("eval_recall"),
#             "train_time": ts.get("total_flos", 0),  # approximate or replace
#             "path": os.path.join(outdir, "final")
#         }

# # append to results_all if found
# results_all = []
# if best_metrics:
#     results_all.append(best_metrics)
#     results_df = pd.DataFrame(results_all)[
#         ["model","eval_accuracy","eval_f1","eval_precision","eval_recall","train_time","path"]
#     ]
#     display(results_df)
# else:
#     print("trainer_state.json not found or no eval logs present.")


In [9]:
# results_df = pd.DataFrame(results_all)[
#     ["model","eval_accuracy","eval_f1","eval_precision","eval_recall","train_time","path"]
# ]
# print("\n🏁 Model Comparison:")
# display(results_df)


In [10]:
# from transformers import pipeline
# import time

# def benchmark_inference(model_path):
#     clf = pipeline("text-classification", model=model_path, tokenizer=model_path)
#     samples = [
#         "The product quality is amazing!",
#         "It's okay, not too bad but not great either.",
#         "Worst experience ever, completely disappointed."
#     ] * 100
#     start = time.time()
#     _ = clf(samples)
#     return round(time.time()-start,2)

# for r in results_all:
#     t = benchmark_inference(r["path"])
#     print(f"{r['model']} inference time (300 texts): {t}s")
